# Pre-SFT Baseline: Context-Parametric Inversion

Establishes the true step-0 anchor for the CPI trajectory: evaluates the raw
pretrained base model (no LoRA adapter -- the exact pre-SFT state both the
`alpaca` and `tulu` runs started from) on the conflict-eval set.

Uses `eval.py --logprob-only`: Layer 0 (filter) + Layer 1 (`method_logprob`)
are both teacher-forced log-prob comparisons over the candidate answers, so
neither needs the model to generate free-form text. A raw base model can't
reliably follow the "answer in a few words" instruction -- `--logprob-only`
skips generation entirely rather than trying to parse noisy generated text
that was never load-bearing for `R_ctx`/`R_par` to begin with.

## 0. Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 1. Get the codebase (`dev` branch)

Clones fresh if not already present; otherwise fetches and fast-forwards to
the latest `dev`. `dev` is where active work lands (the judge-fix-only
correction + this baseline's `--logprob-only` flag are both already there).

In [ ]:
import os

GITHUB_REPO_URL = "https://github.com/GIRIAYUSH/context-parametric-inversion-research.git"
REPO_DIR = "/content/context-parametric-inversion-research"
BRANCH = "dev"

if not os.path.isdir(REPO_DIR):
    !git clone --branch {BRANCH} {GITHUB_REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git fetch origin {BRANCH} && git checkout {BRANCH} && git pull --ff-only origin {BRANCH}

%cd {REPO_DIR}
!git log --oneline -3
print("\nRepo dir:", REPO_DIR)

## 2. Install dependencies

In [ ]:
!pip install -q -U transformers accelerate huggingface_hub

## 3. Configuration -- checkpoint locations on Drive + secrets

`CPI_CKPT_DIR_ALPACA` / `CPI_CKPT_DIR_TULU` point at the same Drive folders
used elsewhere (each directly contains `checkpoint-<step>/` and `final/`) --
not needed for the baseline eval itself (no adapter is loaded), but recorded
here so this notebook is self-contained for whatever comes after the
baseline run.

Secrets are pulled from Colab's secrets manager (the key icon in the left
sidebar) via `userdata.get(...)`, never pasted inline -- this notebook lives
under `experiment-notebooks/` and IS tracked by git, unlike `src/runs.ipynb`.

In [ ]:
import os
from google.colab import userdata
from huggingface_hub import login

# <-- EDIT if your Drive paths differ
os.environ["CPI_CKPT_DIR_ALPACA"] = "/content/drive/MyDrive/Checkpoints - Alpaca - CPI- Analysis/Alpaca_Checkpoints/checkpoints"
os.environ["CPI_CKPT_DIR_TULU"]   = "/content/drive/MyDrive/Checkpoints - TULU - CPI- Analysis/checkpoints"

# Base model both runs actually trained on -- congif.yaml's run.model / models.<name>.hf_id
BASE_MODEL = "meta-llama/Llama-2-7b-hf"  # <-- EDIT if different

# Add HF_TOKEN as a Colab secret first (key icon, left sidebar) -- needed for
# the gated Llama-2 weights.
login(token=userdata.get("HF_TOKEN"))

print("CPI_CKPT_DIR_ALPACA:", os.environ["CPI_CKPT_DIR_ALPACA"])
print("CPI_CKPT_DIR_TULU  :", os.environ["CPI_CKPT_DIR_TULU"])
print("BASE_MODEL         :", BASE_MODEL)

## 4. Run the pre-SFT baseline eval (Layers 0+1 only, no generation)

Loads the raw pretrained base model -- no adapter applied, i.e. the true
step-0 state -- with `use_chat_template=True` (default), matching every
existing checkpoint's recorded `config`. Writes
`results/cpi-results/presft-baseline/Llama-2-7b-hf_eval.json` in the same
schema as `results/phase0-results/model_*/results.json`.

In [ ]:
!python src/evaluation/eval.py \
    --model-id {BASE_MODEL} \
    --dataset dataset/conflict_eval_unified.json \
    --output-dir results/cpi-results/presft-baseline \
    --logprob-only

## 5. Copy the result to Drive (optional, keeps it if the Colab runtime resets)

In [ ]:
DRIVE_RESULTS_DIR = "/content/drive/MyDrive/cpi-presft-baseline"  # <-- EDIT if you want a different Drive spot
!mkdir -p "{DRIVE_RESULTS_DIR}"
!cp -v results/cpi-results/presft-baseline/*.json "{DRIVE_RESULTS_DIR}/"